# サンプル間相関解析と群分離評価

**対応するブログ記事**: [blog/article-05b-data-correlation.md](../blog/article-05b-data-correlation.md) — サンプル間相関解析と群分離評価【論文再現シリーズ #5b】  
**実行順序**: 5b番目  
**所要時間**: 約15分

---

## このNotebookで行うこと

32サンプル間のピアソン相関を計算し、Normal/Tumor群の分離品質を評価します：

- 全サンプル間相関マトリクスの作成
- 群内相関 vs 群間相関の比較
- ペア患者相関の詳細解析
- 相関ヒートマップによる視覚的群分離確認
- 異常サンプルや技術的問題の検出

**⚠️ 前提条件**:
- [notebook_05a_data_distribution.ipynb](./notebook_05a_data_distribution.ipynb) が完了していること
- タンパク質マトリクスとサンプル情報が読み込み済みであること

## 1. ライブラリと設定の確認

前のNotebookから継続して実行する場合は、この部分はスキップできます。

In [ ]:
import numpy as np          # 数値計算ライブラリ（配列操作・統計計算に使用）
import pandas as pd         # データフレーム操作ライブラリ（CSV読み込み・データ操作に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（ヒストグラム・散布図作成に使用）
import seaborn as sns       # 統計的可視化ライブラリ（相関ヒートマップ・分布プロットに使用）
from scipy import stats    # 統計計算ライブラリ（正規性検定・相関解析に使用）
import warnings            # 警告制御（matplotlib警告の非表示用）
warnings.filterwarnings('ignore')  # グラフ描画時の軽微な警告を非表示

# Jupyter notebook用の設定
%matplotlib inline

print("✅ ライブラリの読み込み完了")

In [ ]:
# 前回のNotebookで設定済みの場合はスキップ可能
try:
    # 既に変数が定義されているかチェック
    print(f"📊 既存データ: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")
    print(f"📋 サンプル情報: {len(sample_info)} サンプル")
    print("✅ 前回のデータが利用可能です")
except NameError:
    # データが未読み込みの場合は新たに読み込み
    print("⚠️ データを新たに読み込みます...")
    
    # --- パス設定 ---
    RESULTS_DIR = "../results"
    DATA_FILE = f"{RESULTS_DIR}/protein_matrix_from_sage.csv"
    FIG_DIR = f"{RESULTS_DIR}/figures"
    TABLES_DIR = f"{RESULTS_DIR}/tables"
    
    # --- 色設定 ---
    NORMAL_COLOR = "#3498DB"   # Normal群の色（青）
    TUMOR_COLOR = "#E74C3C"    # Tumor群の色（赤）
    
    # データ読み込み
    def load_protein_matrix_with_metadata():
        df = pd.read_csv(DATA_FILE, index_col=0)
        sample_info_list = []
        for sample in df.columns:
            parts = sample.split("-")
            if len(parts) == 2:
                patient_id = parts[0]
                condition = "Normal" if parts[1] == "N" else "Tumor"
            else:
                patient_id, condition = sample, "Unknown"
            
            sample_info_list.append({
                'Sample': sample,
                'Patient': patient_id,
                'Condition': condition
            })
        
        sample_info = pd.DataFrame(sample_info_list)
        return df, sample_info
    
    df, sample_info = load_protein_matrix_with_metadata()
    print(f"✅ データ読み込み完了: {df.shape[0]:,} タンパク質 × {df.shape[1]} サンプル")

## 2. サンプル間相関解析

### 【評価項目】
1. 全サンプル間のピアソン相関
2. 群内相関 vs 群間相関
3. ペア患者の相関（Normal vs Tumor）
4. 異常サンプルの検出

In [ ]:
def analyze_sample_correlations(df, sample_info):
    """サンプル間の相関を解析し、群分離と技術再現性を評価する。

    【評価項目】
    1. 全サンプル間のピアソン相関
    2. 群内相関 vs 群間相関
    3. ペア患者の相関（Normal vs Tumor）
    4. 異常サンプルの検出
    """
    print("\n=== サンプル間相関解析 ===")

    # 欠損値を含むデータでも相関計算可能にするため、pairwiseでの相関を計算
    # pearson: 線形相関の強さ（-1〜+1、1に近いほど強い正の相関）
    corr_matrix = df.corr(method='pearson')

    # 相関の基本統計
    # 上三角部分のみ取り出す（対角成分と重複を除去）
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    all_correlations = upper_triangle.stack().values  # 上三角部分を1次元配列に変換

    print(f"サンプル間相関統計:")
    print(f"  平均相関: {np.mean(all_correlations):.3f}")
    print(f"  中央値相関: {np.median(all_correlations):.3f}")
    print(f"  標準偏差: {np.std(all_correlations):.3f}")
    print(f"  最小相関: {np.min(all_correlations):.3f}")
    print(f"  最大相関: {np.max(all_correlations):.3f}")

    # 群別相関解析
    normal_samples = sample_info[sample_info['Condition'] == 'Normal']['Sample'].tolist()
    tumor_samples = sample_info[sample_info['Condition'] == 'Tumor']['Sample'].tolist()

    print(f"\n📊 サンプル数の確認:")
    print(f"  Normal群: {len(normal_samples)}")
    print(f"  Tumor群: {len(tumor_samples)}")

    # 群内相関（同じ条件同士）
    normal_corr = corr_matrix.loc[normal_samples, normal_samples]
    tumor_corr = corr_matrix.loc[tumor_samples, tumor_samples]

    # 群間相関（Normal vs Tumor）
    between_corr = corr_matrix.loc[normal_samples, tumor_samples]

    # 上三角部分を取り出して統計計算
    normal_within = normal_corr.where(np.triu(np.ones(normal_corr.shape), k=1).astype(bool)).stack().values
    tumor_within = tumor_corr.where(np.triu(np.ones(tumor_corr.shape), k=1).astype(bool)).stack().values
    between_vals = between_corr.values.flatten()

    print(f"\n群別相関解析:")
    print(f"  Normal群内平均相関: {np.mean(normal_within):.3f}")
    print(f"  Tumor群内平均相関: {np.mean(tumor_within):.3f}")
    print(f"  群間平均相関: {np.mean(between_vals):.3f}")

    # 相関ヒートマップの作成
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # 1. 全体相関ヒートマップ
    # サンプルを条件別に並べ替え
    ordered_samples = normal_samples + tumor_samples
    ordered_corr = corr_matrix.loc[ordered_samples, ordered_samples]

    im1 = ax1.imshow(ordered_corr.values, cmap='RdBu_r', vmin=0.5, vmax=1.0, aspect='equal')
    ax1.set_title("Sample-Sample Correlation Matrix")
    ax1.set_xlabel("Samples")
    ax1.set_ylabel("Samples")

    # Normal/Tumor境界線を表示
    boundary = len(normal_samples) - 0.5
    ax1.axhline(y=boundary, color='black', linewidth=2)
    ax1.axvline(x=boundary, color='black', linewidth=2)

    # ラベル設定（患者IDのみ表示）
    patient_labels = [sample_info[sample_info['Sample'] == s]['Patient'].iloc[0] for s in ordered_samples]
    ax1.set_xticks(range(len(ordered_samples)))
    ax1.set_yticks(range(len(ordered_samples)))
    ax1.set_xticklabels(patient_labels, rotation=45, fontsize=8)
    ax1.set_yticklabels(patient_labels, fontsize=8)

    # カラーバー
    plt.colorbar(im1, ax=ax1, shrink=0.8, label="Pearson Correlation")

    # 2. 相関分布の比較
    ax2.hist(normal_within, bins=20, alpha=0.5, label='Normal-Normal', color=NORMAL_COLOR, density=True)
    ax2.hist(tumor_within, bins=20, alpha=0.5, label='Tumor-Tumor', color=TUMOR_COLOR, density=True)
    ax2.hist(between_vals, bins=20, alpha=0.5, label='Normal-Tumor', color='gray', density=True)
    ax2.set_xlabel("Correlation Coefficient")
    ax2.set_ylabel("Density")
    ax2.set_title("Distribution of Correlations")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/sample_correlation_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 相関統計をCSV保存
    correlation_stats = {
        'Correlation_Type': ['Overall_Mean', 'Normal_Within', 'Tumor_Within', 'Between_Groups'],
        'Mean_Correlation': [
            np.mean(all_correlations),
            np.mean(normal_within),
            np.mean(tumor_within),
            np.mean(between_vals)
        ],
        'Std_Correlation': [
            np.std(all_correlations),
            np.std(normal_within),
            np.std(tumor_within),
            np.std(between_vals)
        ]
    }
    correlation_df = pd.DataFrame(correlation_stats)
    correlation_df.to_csv(f"{TABLES_DIR}/sample_correlation_stats.csv", index=False)
    
    print(f"\n💾 相関統計を保存: {TABLES_DIR}/sample_correlation_stats.csv")
    print(f"🎨 相関図を保存: {FIG_DIR}/sample_correlation_analysis.png")

    return correlation_df

# サンプル間相関解析の実行
correlation_stats = analyze_sample_correlations(df, sample_info)

## 3. 相関統計の詳細確認

In [ ]:
# 相関統計の表示
print("=== 📊 相関解析統計サマリー ===")
display(correlation_stats)

# 群分離品質の評価
normal_within_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Normal_Within']['Mean_Correlation'].iloc[0]
tumor_within_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Tumor_Within']['Mean_Correlation'].iloc[0]
between_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Between_Groups']['Mean_Correlation'].iloc[0]

print(f"\n=== 群分離品質評価 ===")
print(f"📈 相関値の比較:")
print(f"  - Normal群内: {normal_within_corr:.3f}")
print(f"  - Tumor群内: {tumor_within_corr:.3f}")
print(f"  - 群間: {between_corr:.3f}")

# 分離度の計算
within_avg = (normal_within_corr + tumor_within_corr) / 2
separation_score = within_avg - between_corr

print(f"\n✅ 評価結果:")
print(f"  群内平均相関: {within_avg:.3f}")
print(f"  分離スコア: {separation_score:.3f}")

if separation_score > 0.1:
    print(f"  🎉 優良な群分離 (分離スコア > 0.1)")
elif separation_score > 0.05:
    print(f"  ✅ 良好な群分離 (分離スコア > 0.05)")
else:
    print(f"  ⚠️ 群分離要注意 (分離スコア < 0.05)")

if within_avg > 0.8:
    print(f"  ✅ 高い技術再現性 (群内相関 > 0.8)")
elif within_avg > 0.7:
    print(f"  ✅ 良好な技術再現性 (群内相関 > 0.7)")
else:
    print(f"  ⚠️ 技術再現性要注意 (群内相関 < 0.7)")

## 4. ペア患者相関の詳細解析

### 【ペア解析の意義】
同一患者内でのNormal vs Tumorの相関が、異なる患者間の相関より高いかを確認します。

In [ ]:
def analyze_paired_patient_correlations(df, sample_info):
    """同一患者のNormal/Tumor間相関を詳細解析する。

    【ペア解析の意義】
    同一患者内でのNormal vs Tumorの相関が、
    異なる患者間の相関より高いかを確認する。
    """
    print("\n=== ペア患者相関解析 ===")

    paired_correlations = []
    unpaired_correlations = []

    # 患者別にペア相関を計算
    for patient_id in sample_info['Patient'].unique():
        patient_samples = sample_info[sample_info['Patient'] == patient_id]['Sample'].tolist()

        if len(patient_samples) == 2:  # Normal/Tumorペアが存在
            normal_sample = [s for s in patient_samples if 'N' in s]
            tumor_sample = [s for s in patient_samples if 'T' in s]

            if normal_sample and tumor_sample:
                # 同一患者内のNormal vs Tumor相関
                pair_corr = df[normal_sample[0]].corr(df[tumor_sample[0]], method='pearson')
                if not np.isnan(pair_corr):  # NaNでない場合のみ追加
                    paired_correlations.append({
                        'Patient': patient_id,
                        'Normal_Sample': normal_sample[0],
                        'Tumor_Sample': tumor_sample[0],
                        'Correlation': pair_corr
                    })

    # ペア相関の統計
    pair_corrs = [p['Correlation'] for p in paired_correlations]

    print(f"ペア患者数: {len(pair_corrs)}")
    if len(pair_corrs) > 0:
        print(f"ペア内平均相関: {np.mean(pair_corrs):.3f}")
        print(f"ペア内相関範囲: {np.min(pair_corrs):.3f} - {np.max(pair_corrs):.3f}")
        print(f"ペア内標準偏差: {np.std(pair_corrs):.3f}")
    else:
        print("⚠️ ペア患者が見つかりませんでした")
        return pd.DataFrame()

    # ペア相関の可視化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # 1. ペア相関のヒストグラム
    ax1.hist(pair_corrs, bins=10, alpha=0.7, color='purple', edgecolor='black')
    ax1.axvline(np.mean(pair_corrs), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {np.mean(pair_corrs):.3f}')
    ax1.set_xlabel("Correlation Coefficient")
    ax1.set_ylabel("Frequency")
    ax1.set_title("Normal-Tumor Correlation within Patients")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # 2. ペア相関 vs 群間相関の比較
    between_mean = correlation_stats[correlation_stats['Correlation_Type'] == 'Between_Groups']['Mean_Correlation'].iloc[0]

    # Box plot for comparison
    box_data = [pair_corrs, [between_mean] * len(pair_corrs)]
    ax2.boxplot(box_data, labels=['Within Patient', 'Between Groups'])
    ax2.set_ylabel("Correlation Coefficient")
    ax2.set_title("Paired vs Unpaired Correlations")
    ax2.grid(True, alpha=0.3)

    # 統計的差の表示
    mean_diff = np.mean(pair_corrs) - between_mean
    ax2.text(0.5, 0.95, f'Difference: {mean_diff:.3f}', 
             transform=ax2.transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/paired_patient_correlation.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ペア相関データをCSV保存
    paired_df = pd.DataFrame(paired_correlations)
    paired_df.to_csv(f"{TABLES_DIR}/paired_patient_correlations.csv", index=False)
    
    print(f"\n💾 ペア相関データを保存: {TABLES_DIR}/paired_patient_correlations.csv")
    print(f"🎨 ペア相関図を保存: {FIG_DIR}/paired_patient_correlation.png")

    return paired_df

# ペア患者相関解析の実行
paired_correlations = analyze_paired_patient_correlations(df, sample_info)

## 5. ペア相関の詳細分析

In [ ]:
if len(paired_correlations) > 0:
    print("=== 📊 ペア患者相関詳細 ===")
    display(paired_correlations)
    
    # 個別患者の相関値分析
    pair_corrs = paired_correlations['Correlation'].values
    between_mean = correlation_stats[correlation_stats['Correlation_Type'] == 'Between_Groups']['Mean_Correlation'].iloc[0]
    
    print(f"\n=== ペア vs 群間比較 ===")
    print(f"📈 ペア内平均相関: {np.mean(pair_corrs):.3f}")
    print(f"📉 群間平均相関: {between_mean:.3f}")
    print(f"📊 差分: {np.mean(pair_corrs) - between_mean:.3f}")
    
    # 個体差 vs 疾患効果の評価
    if np.mean(pair_corrs) > between_mean:
        print(f"\n✅ 解釈: 同一患者内でのNormal-Tumor相関が群間相関より高い")
        print(f"   → 個体差よりも疾患状態による変化が大きい")
        print(f"   → 生物学的に妥当なパターン")
    else:
        print(f"\n⚠️ 解釈: ペア内相関が群間相関より低い")
        print(f"   → 個体差が疾患効果より大きい可能性")
        print(f"   → データ品質または実験デザインの検討が必要")
    
    # 相関値の分布を詳細分析
    high_corr_pairs = len([c for c in pair_corrs if c > 0.8])
    medium_corr_pairs = len([c for c in pair_corrs if 0.6 <= c <= 0.8])
    low_corr_pairs = len([c for c in pair_corrs if c < 0.6])
    
    print(f"\n📊 ペア相関分布:")
    print(f"  高相関 (>0.8): {high_corr_pairs} ペア ({high_corr_pairs/len(pair_corrs)*100:.1f}%)")
    print(f"  中相関 (0.6-0.8): {medium_corr_pairs} ペア ({medium_corr_pairs/len(pair_corrs)*100:.1f}%)")
    print(f"  低相関 (<0.6): {low_corr_pairs} ペア ({low_corr_pairs/len(pair_corrs)*100:.1f}%)")

else:
    print("❌ ペア患者データが不足しているため、詳細分析をスキップします")

## 6. 相関マトリクスの詳細解析

In [ ]:
def create_enhanced_correlation_heatmap(df, sample_info):
    """より詳細な相関ヒートマップを作成する。"""
    print("\n=== 拡張相関ヒートマップ作成 ===")
    
    # 相関マトリクスの再計算
    corr_matrix = df.corr(method='pearson')
    
    # サンプルを条件と患者IDで並べ替え
    normal_samples = sample_info[sample_info['Condition'] == 'Normal']['Sample'].tolist()
    tumor_samples = sample_info[sample_info['Condition'] == 'Tumor']['Sample'].tolist()
    
    # 患者IDでソート
    def get_patient_id(sample):
        return sample_info[sample_info['Sample'] == sample]['Patient'].iloc[0]
    
    normal_samples_sorted = sorted(normal_samples, key=get_patient_id)
    tumor_samples_sorted = sorted(tumor_samples, key=get_patient_id)
    
    ordered_samples = normal_samples_sorted + tumor_samples_sorted
    ordered_corr = corr_matrix.loc[ordered_samples, ordered_samples]
    
    # 拡張ヒートマップ
    plt.figure(figsize=(16, 12))
    
    # seabornのclustermapを使用（デンドログラム付き）
    # ただし、row_clusterとcol_clusterをFalseにして順序を保持
    g = sns.clustermap(ordered_corr, 
                       cmap='RdBu_r', 
                       vmin=0.3, vmax=1.0,
                       row_cluster=False, col_cluster=False,
                       figsize=(14, 12),
                       cbar_kws={'label': 'Pearson Correlation'})
    
    # タイトルとラベル設定
    g.ax_heatmap.set_title('Sample-Sample Correlation Matrix\n(Ordered by Condition and Patient)', 
                          fontsize=16, pad=20)
    
    # 患者IDラベルを設定
    patient_labels = [get_patient_id(s) for s in ordered_samples]
    condition_labels = [sample_info[sample_info['Sample'] == s]['Condition'].iloc[0] for s in ordered_samples]
    
    # ラベルに条件を追加
    combined_labels = [f"{p}({c[0]})" for p, c in zip(patient_labels, condition_labels)]
    
    g.ax_heatmap.set_xticklabels(combined_labels, rotation=45, fontsize=8)
    g.ax_heatmap.set_yticklabels(combined_labels, rotation=0, fontsize=8)
    
    # Normal/Tumor境界線
    boundary = len(normal_samples_sorted) - 0.5
    g.ax_heatmap.axhline(y=boundary, color='black', linewidth=3)
    g.ax_heatmap.axvline(x=boundary, color='black', linewidth=3)
    
    plt.savefig(f"{FIG_DIR}/enhanced_correlation_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print(f"🎨 拡張ヒートマップを保存: {FIG_DIR}/enhanced_correlation_heatmap.png")
    
    return ordered_corr

# 拡張ヒートマップの作成
enhanced_corr = create_enhanced_correlation_heatmap(df, sample_info)

## 7. 結果のまとめとファイル保存

In [ ]:
# 相関解析の総合サマリー
print("=== 📋 相関解析 総合結果 ===")

# 主要統計値の抽出
normal_within_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Normal_Within']['Mean_Correlation'].iloc[0]
tumor_within_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Tumor_Within']['Mean_Correlation'].iloc[0]
between_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Between_Groups']['Mean_Correlation'].iloc[0]
overall_corr = correlation_stats[correlation_stats['Correlation_Type'] == 'Overall_Mean']['Mean_Correlation'].iloc[0]

print(f"\n📊 相関統計サマリー:")
print(f"  全体平均相関: {overall_corr:.3f}")
print(f"  Normal群内平均: {normal_within_corr:.3f}")
print(f"  Tumor群内平均: {tumor_within_corr:.3f}")
print(f"  群間平均: {between_corr:.3f}")

if len(paired_correlations) > 0:
    pair_corrs = paired_correlations['Correlation'].values
    print(f"  ペア内平均: {np.mean(pair_corrs):.3f}")

# データ品質の総合評価
within_avg = (normal_within_corr + tumor_within_corr) / 2
separation_score = within_avg - between_corr

print(f"\n✅ データ品質評価:")
print(f"  群内一貫性: {'優良' if within_avg > 0.8 else '良好' if within_avg > 0.7 else '要改善'} ({within_avg:.3f})")
print(f"  群分離度: {'優良' if separation_score > 0.1 else '良好' if separation_score > 0.05 else '要注意'} ({separation_score:.3f})")

if len(paired_correlations) > 0:
    individual_vs_disease = np.mean(pair_corrs) > between_corr
    print(f"  疾患効果: {'検出可能' if individual_vs_disease else '個体差大'}")

print(f"\n🎯 統計解析への推奨:")
if within_avg > 0.8 and separation_score > 0.05:
    print(f"  ✅ 統計解析に適したデータ品質")
    print(f"  ✅ t検定・ANOVAの実行が推奨")
    print(f"  ✅ 多変量解析（PCA・クラスタリング）も有効")
else:
    print(f"  ⚠️ 追加の前処理・品質管理が推奨")
    
# 保存されたファイルの一覧表示
print(f"\n💾 生成されたファイル:")
print(f"  📊 相関統計: {TABLES_DIR}/sample_correlation_stats.csv")
if len(paired_correlations) > 0:
    print(f"  👥 ペア相関: {TABLES_DIR}/paired_patient_correlations.csv")
print(f"  🎨 基本相関図: {FIG_DIR}/sample_correlation_analysis.png")
if len(paired_correlations) > 0:
    print(f"  📈 ペア相関図: {FIG_DIR}/paired_patient_correlation.png")
print(f"  🔍 拡張ヒートマップ: {FIG_DIR}/enhanced_correlation_heatmap.png")

## 📚 このNotebookで学んだこと

### 相関解析の要点

1. **群分離の確認**: Normal/Tumor群が相関パターンで明確に分離されることを確認
2. **技術的再現性**: 群内相関が高く、技術的な一貫性が確保されている
3. **生物学的妥当性**: ペア患者解析により個体差と疾患効果を区別
4. **統計解析適性**: データが統計的解析に適した品質であることを確認

### 重要な評価指標

| 指標 | 良好な基準 | 意味 |
|------|------------|------|
| **群内相関** | >0.8 | 技術的再現性・サンプル品質 |
| **群間相関** | <0.7 | 生物学的差異の検出可能性 |
| **分離スコア** | >0.05 | 群分離の明確さ |
| **ペア内相関** | >群間相関 | 疾患効果 vs 個体差 |

### 相関パターンの解釈

- **高い群内相関**: サンプル処理・測定の技術的安定性が高い
- **低い群間相関**: Normal/Tumor間で生物学的差異が存在
- **ペア内相関 > 群間相関**: 疾患による変化が個体差より大きい

---

**次のステップ**: 外れ値検出と総合品質評価でデータクリーニングの方針を決定します

➡️ **次回**: [notebook_05c_data_summary.ipynb](./notebook_05c_data_summary.ipynb) — 総合品質評価と外れ値検出